In [8]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully.")

Libraries imported successfully.


In [9]:
PROJECT_ROOT = Path.cwd().parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"

print(f"Project root: {PROJECT_ROOT}")

Project root: c:\Users\Abhishek\Desktop\CloudSentinel-AI


In [10]:
V2_DATA_FILE = PROCESSED_DATA_DIR / "cloudsentinel_v2_predictions.csv"

feature_df = pd.read_csv(
    V2_DATA_FILE,
    parse_dates=["timestamp"]
)

print("V2 dataset loaded successfully!")
print(f"Shape: {feature_df.shape}")

V2 dataset loaded successfully!
Shape: (14400, 22)


In [11]:
feature_df.head()

,timestamp,resource_id,resource_type,cpu_utilization,memory_utilization,network_in,request_count,storage_usage,estimated_cost,is_anomaly,...,hour,day_of_week,is_weekend,resource_utilization,cpu_memory_ratio,network_per_request,requests_per_network,cost_per_request,cost_per_cpu,ml_prediction_v2
0,2026-08-01 00:00:00,EC2-001,Web Server,19.087236,24.906175,74.708176,899.351067,54.324431,0.059451,0,...,0,5,1,32.772614,0.766366,0.082977,11.879180,0.000066,0.003115,0
1,2026-08-01 00:15:00,EC2-001,Web Server,21.372462,28.249815,53.193879,739.607109,56.138281,0.060514,0,...,0,5,1,35.253519,0.756552,0.071825,13.647429,0.000082,0.002831,0
2,2026-08-01 00:30:00,EC2-001,Web Server,20.178249,26.371394,57.415241,773.460620,54.555520,0.059651,0,...,0,5,1,33.701721,0.765157,0.074136,13.240733,0.000077,0.002956,0
3,2026-08-01 00:45:00,EC2-001,Web Server,16.712111,24.016472,77.400133,922.063367,53.726745,0.058352,0,...,0,5,1,31.485109,0.695860,0.083851,11.760992,0.000063,0.003492,0
4,2026-08-01 01:00:00,EC2-001,Web Server,19.658934,25.695524,63.384328,790.523342,54.572166,0.059368,0,...,1,5,1,33.308874,0.765072,0.080079,12.278195,0.000075,0.003020,0


In [12]:
V2_DATA_FILE = PROCESSED_DATA_DIR / "cloudsentinel_v2_predictions.csv"

print("Saved dataset exists:", V2_DATA_FILE.exists())

Saved dataset exists: True


In [13]:
feature_df = pd.read_csv(
    V2_DATA_FILE,
    parse_dates=["timestamp"]
)

print("Dataset loaded successfully!")
print("Shape:", feature_df.shape)
print("\nColumns:")
print(feature_df.columns.tolist())

Dataset loaded successfully!
Shape: (14400, 22)

Columns:
['timestamp', 'resource_id', 'resource_type', 'cpu_utilization', 'memory_utilization', 'network_in', 'request_count', 'storage_usage', 'estimated_cost', 'is_anomaly', 'anomaly_type', 'ml_prediction', 'hour', 'day_of_week', 'is_weekend', 'resource_utilization', 'cpu_memory_ratio', 'network_per_request', 'requests_per_network', 'cost_per_request', 'cost_per_cpu', 'ml_prediction_v2']


In [14]:
# Create a separate dataset for V3
v3_df = feature_df.copy()

baseline_features = [
    "cpu_utilization",
    "memory_utilization",
    "network_in",
    "request_count",
    "estimated_cost"
]

# Calculate each resource's typical value for each metric
for col in baseline_features:
    resource_mean = v3_df.groupby("resource_id")[col].transform("mean")
    resource_std = v3_df.groupby("resource_id")[col].transform("std")

    # Standardized deviation from the resource's typical behavior
    v3_df[f"{col}_deviation"] = (
        (v3_df[col] - resource_mean)
        / (resource_std + 1e-6)
    )

print("Resource-aware deviation features created!")

v3_df[
    [
        "resource_id",
        "cpu_utilization",
        "cpu_utilization_deviation",
        "estimated_cost",
        "estimated_cost_deviation"
    ]
].head()

Resource-aware deviation features created!


,resource_id,cpu_utilization,cpu_utilization_deviation,estimated_cost,estimated_cost_deviation
0,EC2-001,19.087236,-1.260706,0.059451,-1.143507
1,EC2-001,21.372462,-1.009694,0.060514,-1.045272
2,EC2-001,20.178249,-1.140868,0.059651,-1.125073
3,EC2-001,16.712111,-1.521593,0.058352,-1.245080
4,EC2-001,19.658934,-1.197910,0.059368,-1.151183


In [15]:
deviation_columns = [
    f"{col}_deviation"
    for col in baseline_features
]

v3_df[deviation_columns].describe()

,cpu_utilization_deviation,memory_utilization_deviation,network_in_deviation,request_count_deviation,estimated_cost_deviation
count,1.440000e+04,1.440000e+04,1.440000e+04,1.440000e+04,1.440000e+04
mean,1.554312e-17,1.233581e-17,7.648203e-17,6.315935e-17,-1.776357e-16
std,9.998610e-01,9.998610e-01,9.998611e-01,9.998611e-01,9.997732e-01
min,-2.175697e+00,-1.863316e+00,-1.982266e+00,-2.396357e+00,-1.536232e+00
25%,-8.924065e-01,-9.802827e-01,-5.431301e-01,-9.142587e-01,-8.265561e-01
50%,2.206136e-01,4.199136e-01,7.611513e-02,2.516260e-01,3.210623e-01
75%,8.004337e-01,8.440918e-01,4.415314e-01,8.432173e-01,6.721547e-01
max,8.081676e+00,5.231503e+00,2.104580e+01,8.041152e+00,1.787249e+01


In [16]:
original_features = [
    "cpu_utilization",
    "memory_utilization",
    "network_in",
    "request_count",
    "storage_usage",
    "estimated_cost"
]

deviation_columns = [
    "cpu_utilization_deviation",
    "memory_utilization_deviation",
    "network_in_deviation",
    "request_count_deviation",
    "estimated_cost_deviation"
]

feature_columns_v3 = original_features + deviation_columns

X_v3 = v3_df[feature_columns_v3].copy()

print("V3 feature matrix shape:", X_v3.shape)
print("Missing values:", X_v3.isnull().sum().sum())
print("Infinite values:", np.isinf(X_v3.to_numpy()).sum())

V3 feature matrix shape: (14400, 11)
Missing values: 0
Infinite values: 0


In [17]:
model_v3 = IsolationForest(
    n_estimators=200,
    contamination=0.007,
    random_state=42,
    n_jobs=-1
)

model_v3.fit(X_v3)

print("Isolation Forest V3 trained successfully!")

Isolation Forest V3 trained successfully!


In [18]:
predictions_v3 = model_v3.predict(X_v3)

v3_df["ml_prediction_v3"] = np.where(
    predictions_v3 == -1,
    1,
    0
)

print("V3 prediction counts:")
print(v3_df["ml_prediction_v3"].value_counts())

V3 prediction counts:
ml_prediction_v3
0    14299
1      101
Name: count, dtype: int64


In [19]:
y_true = v3_df["is_anomaly"]
y_pred_v3 = v3_df["ml_prediction_v3"]

precision_v3 = precision_score(y_true, y_pred_v3, zero_division=0)
recall_v3 = recall_score(y_true, y_pred_v3, zero_division=0)
f1_v3 = f1_score(y_true, y_pred_v3, zero_division=0)

print(f"V3 Precision: {precision_v3:.4f}")
print(f"V3 Recall:    {recall_v3:.4f}")
print(f"V3 F1 Score:  {f1_v3:.4f}")

V3 Precision: 0.5743
V3 Recall:    0.5800
V3 F1 Score:  0.5771


In [21]:
pd.crosstab(
    v3_df["anomaly_type"],
    v3_df["ml_prediction_v3"],
    rownames=["Actual Anomaly Type"],
    colnames=["V3 Prediction"]
)

V3 Prediction,0,1
Actual Anomaly Type,,
combined_spike,0,15
cost_spike,6,14
cpu_spike,13,12
memory_spike,18,2
network_spike,5,15
normal,14257,43


In [22]:
V3_DATA_FILE = PROCESSED_DATA_DIR / "cloudsentinel_v3_predictions.csv"

v3_df.to_csv(V3_DATA_FILE, index=False)

print(f"V3 predictions saved to: {V3_DATA_FILE}")

V3 predictions saved to: c:\Users\Abhishek\Desktop\CloudSentinel-AI\data\processed\cloudsentinel_v3_predictions.csv


In [23]:
RAW_DATA_FILE = RAW_DATA_DIR / "cloud_resource_metrics.csv"

raw_df = pd.read_csv(
    RAW_DATA_FILE,
    parse_dates=["timestamp"]
)

# Sort chronologically across all resources
raw_df = raw_df.sort_values(
    ["timestamp", "resource_id"]
).reset_index(drop=True)

print("Dataset shape:", raw_df.shape)
print("Start:", raw_df["timestamp"].min())
print("End:", raw_df["timestamp"].max())

Dataset shape: (14400, 11)
Start: 2026-08-01 00:00:00
End: 2026-08-30 23:45:00


In [24]:
unique_timestamps = np.sort(
    raw_df["timestamp"].unique()
)

split_index = int(len(unique_timestamps) * 0.8)
split_timestamp = pd.Timestamp(
    unique_timestamps[split_index]
)

train_df = raw_df[
    raw_df["timestamp"] < split_timestamp
].copy()

test_df = raw_df[
    raw_df["timestamp"] >= split_timestamp
].copy()

print("Split timestamp:", split_timestamp)
print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)

print("\nTraining anomaly counts:")
print(train_df["is_anomaly"].value_counts())

print("\nTesting anomaly counts:")
print(test_df["is_anomaly"].value_counts())

Split timestamp: 2026-08-25 00:00:00
Training shape: (11520, 11)
Testing shape: (2880, 11)

Training anomaly counts:
is_anomaly
0    11437
1       83
Name: count, dtype: int64

Testing anomaly counts:
is_anomaly
0    2863
1      17
Name: count, dtype: int64


In [25]:
baseline_features = [
    "cpu_utilization",
    "memory_utilization",
    "network_in",
    "request_count",
    "estimated_cost"
]

resource_stats = train_df.groupby("resource_id")[baseline_features].agg(
    ["mean", "std"]
)

def add_deviation_features(data, stats):
    result = data.copy()

    for col in baseline_features:
        means = result["resource_id"].map(stats[(col, "mean")])
        stds = result["resource_id"].map(stats[(col, "std")])

        result[f"{col}_deviation"] = (
            (result[col] - means) / (stds + 1e-6)
        )

    return result

train_features = add_deviation_features(train_df, resource_stats)
test_features = add_deviation_features(test_df, resource_stats)

print("Training features:", train_features.shape)
print("Testing features:", test_features.shape)
print("Missing training values:", train_features.isnull().sum().sum())
print("Missing testing values:", test_features.isnull().sum().sum())

Training features: (11520, 16)
Testing features: (2880, 16)
Missing training values: 0
Missing testing values: 0


In [26]:
original_features = [
    "cpu_utilization",
    "memory_utilization",
    "network_in",
    "request_count",
    "storage_usage",
    "estimated_cost"
]

deviation_columns = [
    f"{col}_deviation"
    for col in baseline_features
]

feature_columns = original_features + deviation_columns

X_train = train_features[feature_columns].copy()
X_test = test_features[feature_columns].copy()

y_test = test_df["is_anomaly"].to_numpy()

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Test anomalies:", y_test.sum())

X_train: (11520, 11)
X_test: (2880, 11)
Test anomalies: 17


In [27]:
from sklearn.ensemble import IsolationForest

validation_model = IsolationForest(
    n_estimators=200,
    contamination=0.007,
    random_state=42,
    n_jobs=-1
)

validation_model.fit(X_train)

print("Chronological training completed!")

Chronological training completed!


In [28]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

test_predictions = np.where(
    validation_model.predict(X_test) == -1,
    1,
    0
)

print("Predicted anomaly counts:")
print(pd.Series(test_predictions).value_counts().sort_index())

Predicted anomaly counts:
0    2860
1      20
Name: count, dtype: int64


In [29]:
print("Chronological Validation Results")
print("-" * 35)

print(
    f"Precision: {precision_score(y_test, test_predictions, zero_division=0):.4f}"
)
print(
    f"Recall:    {recall_score(y_test, test_predictions, zero_division=0):.4f}"
)
print(
    f"F1 Score:  {f1_score(y_test, test_predictions, zero_division=0):.4f}"
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_predictions))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        test_predictions,
        labels=[0, 1],
        target_names=["Normal", "Anomaly"],
        zero_division=0
    )
)

Chronological Validation Results
-----------------------------------
Precision: 0.5500
Recall:    0.6471
F1 Score:  0.5946

Confusion Matrix:
[[2854    9]
 [   6   11]]

Classification Report:
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00      2863
     Anomaly       0.55      0.65      0.59        17

    accuracy                           0.99      2880
   macro avg       0.77      0.82      0.80      2880
weighted avg       1.00      0.99      1.00      2880



In [30]:
test_results = test_df.copy()

test_results["predicted_anomaly"] = test_predictions

false_negatives = test_results[
    (test_results["is_anomaly"] == 1) &
    (test_results["predicted_anomaly"] == 0)
]

false_positives = test_results[
    (test_results["is_anomaly"] == 0) &
    (test_results["predicted_anomaly"] == 1)
]

print("Missed anomalies:", len(false_negatives))
print("False alarms:", len(false_positives))

print("\nMissed anomaly types:")
print(false_negatives["anomaly_type"].value_counts())

print("\nFalse alarm resource counts:")
print(false_positives["resource_id"].value_counts())

Missed anomalies: 6
False alarms: 9

Missed anomaly types:
anomaly_type
cpu_spike       3
memory_spike    2
cost_spike      1
Name: count, dtype: int64

False alarm resource counts:
resource_id
EC2-005    4
EC2-003    3
EC2-004    2
Name: count, dtype: int64


In [31]:
columns_to_inspect = [
    "timestamp",
    "resource_id",
    "resource_type",
    "cpu_utilization",
    "memory_utilization",
    "network_in",
    "request_count",
    "estimated_cost",
    "anomaly_type"
]

false_negatives[columns_to_inspect]

,timestamp,resource_id,resource_type,cpu_utilization,memory_utilization,network_in,request_count,estimated_cost,anomaly_type
11693,2026-08-25 08:30:00,EC2-004,Database Server,35.041189,90.370739,105.057444,948.238068,0.074729,memory_spike
11848,2026-08-25 16:15:00,EC2-004,Database Server,57.396423,91.481579,119.683285,1277.950701,0.092647,memory_spike
12121,2026-08-26 06:00:00,EC2-002,Web Server,34.647555,37.829939,116.088023,1472.942747,0.274413,cost_spike
12418,2026-08-26 20:45:00,EC2-004,Database Server,97.687846,71.405367,128.538350,1427.416036,0.088255,cpu_spike
12874,2026-08-27 19:30:00,EC2-005,Worker Server,94.858012,47.549198,94.593952,756.315801,0.070037,cpu_spike
14305,2026-08-30 19:15:00,EC2-001,Web Server,93.315523,55.311856,133.112193,1562.351054,0.077828,cpu_spike


In [32]:
focused_features = [
    "cpu_utilization_deviation",
    "memory_utilization_deviation",
    "network_in_deviation",
    "request_count_deviation",
    "estimated_cost_deviation",
    "estimated_cost",
    "cpu_utilization",
    "memory_utilization"
]

X_train_focused = train_features[focused_features].copy()
X_test_focused = test_features[focused_features].copy()

print("Focused training shape:", X_train_focused.shape)
print("Focused testing shape:", X_test_focused.shape)

Focused training shape: (11520, 8)
Focused testing shape: (2880, 8)


In [33]:
focused_model = IsolationForest(
    n_estimators=200,
    contamination=0.007,
    random_state=42,
    n_jobs=-1
)

focused_model.fit(X_train_focused)

focused_predictions = np.where(
    focused_model.predict(X_test_focused) == -1,
    1,
    0
)

print("Focused model trained and predictions generated.")

Focused model trained and predictions generated.


In [34]:
focused_precision = precision_score(
    y_test, focused_predictions, zero_division=0
)

focused_recall = recall_score(
    y_test, focused_predictions, zero_division=0
)

focused_f1 = f1_score(
    y_test, focused_predictions, zero_division=0
)

print(f"Focused Precision: {focused_precision:.4f}")
print(f"Focused Recall:    {focused_recall:.4f}")
print(f"Focused F1 Score:  {focused_f1:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, focused_predictions))

Focused Precision: 0.5238
Focused Recall:    0.6471
Focused F1 Score:  0.5789

Confusion Matrix:
[[2853   10]
 [   6   11]]


In [35]:
import joblib
import json

MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_path = MODEL_DIR / "cloudsentinel_isolation_forest_v3.joblib"

joblib.dump(validation_model, model_path)

print("Model saved successfully!")
print("Location:", model_path)

Model saved successfully!
Location: c:\Users\Abhishek\Desktop\CloudSentinel-AI\models\cloudsentinel_isolation_forest_v3.joblib


In [36]:
model_metadata = {
    "project": "CloudSentinel AI",
    "model_name": "Isolation Forest V3",
    "model_type": "unsupervised_anomaly_detection",
    "feature_columns": feature_columns,
    "contamination": 0.007,
    "n_estimators": 200,
    "random_state": 42,
    "validation_method": "chronological_holdout",
    "test_precision": 0.5500,
    "test_recall": 0.6471,
    "test_f1": 0.5946,
    "test_records": int(len(y_test)),
    "test_anomalies": int(y_test.sum())
}

metadata_path = MODEL_DIR / "cloudsentinel_v3_metadata.json"

with open(metadata_path, "w") as f:
    json.dump(model_metadata, f, indent=4)

print("Model metadata saved:", metadata_path)

Model metadata saved: c:\Users\Abhishek\Desktop\CloudSentinel-AI\models\cloudsentinel_v3_metadata.json


In [37]:
loaded_model = joblib.load(model_path)

reloaded_predictions = np.where(
    loaded_model.predict(X_test) == -1,
    1,
    0
)

print(
    "Predictions identical after reload:",
    np.array_equal(test_predictions, reloaded_predictions)
)

Predictions identical after reload: True


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score, f1_score

PROJECT_ROOT = Path.cwd().parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Environment restored.")
print("Project root:", PROJECT_ROOT)

Environment restored.
Project root: c:\Users\Abhishek\Desktop\CloudSentinel-AI


In [2]:
RAW_DATA_FILE = RAW_DATA_DIR / "cloud_resource_metrics.csv"

raw_df = pd.read_csv(
    RAW_DATA_FILE,
    parse_dates=["timestamp"]
)

raw_df = raw_df.sort_values(
    ["timestamp", "resource_id"]
).reset_index(drop=True)

unique_timestamps = np.sort(raw_df["timestamp"].unique())

split_index = int(len(unique_timestamps) * 0.8)

split_timestamp = pd.Timestamp(
    unique_timestamps[split_index]
)

train_df = raw_df[
    raw_df["timestamp"] < split_timestamp
].copy()

test_df = raw_df[
    raw_df["timestamp"] >= split_timestamp
].copy()

print("Training records:", len(train_df))
print("Testing records:", len(test_df))
print("Split timestamp:", split_timestamp)

Training records: 11520
Testing records: 2880
Split timestamp: 2026-08-25 00:00:00


In [3]:
baseline_features = [
    "cpu_utilization",
    "memory_utilization",
    "network_in",
    "request_count",
    "estimated_cost"
]

resource_stats = train_df.groupby("resource_id")[
    baseline_features
].agg(["mean", "std"])

print("Training-only resource baselines reconstructed.")
print(resource_stats.head())

Training-only resource baselines reconstructed.
            cpu_utilization            memory_utilization             \
                       mean        std               mean        std   
resource_id                                                            
EC2-001           30.618921   9.120997          43.652077  11.990705   
EC2-002           34.913816  10.218522          45.382761  12.402273   
EC2-003           39.460098  11.207398          50.532728  13.877837   
EC2-004           43.695613  12.246736          59.334591  16.057441   
EC2-005           26.468674   8.678782          39.331522  11.027230   

             network_in            request_count             estimated_cost  \
                   mean        std          mean         std           mean   
resource_id                                                                   
EC2-001      107.230684  56.028086   1307.374381  371.420136       0.071923   
EC2-002      124.844209  64.101294   1567.198463  427.83558

In [4]:
original_features = [
    "cpu_utilization",
    "memory_utilization",
    "network_in",
    "request_count",
    "storage_usage",
    "estimated_cost"
]

deviation_columns = [
    f"{col}_deviation"
    for col in baseline_features
]

feature_columns = original_features + deviation_columns

print("Feature count:", len(feature_columns))
print(feature_columns)

Feature count: 11
['cpu_utilization', 'memory_utilization', 'network_in', 'request_count', 'storage_usage', 'estimated_cost', 'cpu_utilization_deviation', 'memory_utilization_deviation', 'network_in_deviation', 'request_count_deviation', 'estimated_cost_deviation']


In [5]:
model_path = MODEL_DIR / "cloudsentinel_isolation_forest_v3.joblib"

validation_model = joblib.load(model_path)

print("Saved V3 model loaded successfully!")

Saved V3 model loaded successfully!


In [6]:
BASELINE_PATH = MODEL_DIR / "cloudsentinel_resource_baselines_v3.joblib"

INFERENCE_CONFIG_PATH = MODEL_DIR / "cloudsentinel_v3_inference_config.joblib"

joblib.dump(resource_stats, BASELINE_PATH)

inference_config = {
    "feature_columns": feature_columns,
    "baseline_features": baseline_features,
    "original_features": original_features,
    "contamination": 0.007,
    "model_version": "v3"
}

joblib.dump(inference_config, INFERENCE_CONFIG_PATH)

print("Resource baselines saved!")
print("Inference configuration saved!")

Resource baselines saved!
Inference configuration saved!


In [7]:
required_artifacts = [
    model_path,
    BASELINE_PATH,
    INFERENCE_CONFIG_PATH
]

for artifact in required_artifacts:
    print(
        f"{artifact.name}: "
        f"{'OK' if artifact.exists() else 'MISSING'}"
    )

cloudsentinel_isolation_forest_v3.joblib: OK
cloudsentinel_resource_baselines_v3.joblib: OK
cloudsentinel_v3_inference_config.joblib: OK


In [8]:

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from scripts.inference import predict_anomalies

# Use five records from the existing test dataset.
sample_metrics = test_df.head(5).copy()

results = predict_anomalies(sample_metrics)

display(
    results[
        [
            "timestamp",
            "resource_id",
            "cpu_utilization",
            "memory_utilization",
            "estimated_cost",
            "prediction_label",
            "anomaly_score"
        ]
    ]
)


,timestamp,resource_id,cpu_utilization,memory_utilization,estimated_cost,prediction_label,anomaly_score
11520,2026-08-25,EC2-001,20.214634,30.167134,0.061932,normal,-0.125935
11521,2026-08-25,EC2-002,20.297864,27.875231,0.061309,normal,-0.140697
11522,2026-08-25,EC2-003,30.061712,28.258784,0.067164,normal,-0.089333
11523,2026-08-25,EC2-004,25.487776,39.398864,0.064533,normal,-0.065336
11524,2026-08-25,EC2-005,14.886968,26.050064,0.056623,normal,-0.087806


In [9]:

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# Run the saved inference pipeline on the entire held-out test set.
test_results = predict_anomalies(test_df)

y_true = test_results["is_anomaly"].to_numpy()
y_pred = test_results["is_anomaly_prediction"].to_numpy()

print("CloudSentinel AI — V3 Inference Validation")
print("-" * 45)
print("Test records:", len(test_results))
print("Actual anomalies:", int(y_true.sum()))
print("Predicted anomalies:", int(y_pred.sum()))
print(f"Precision: {precision_score(y_true, y_pred, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_true, y_pred, zero_division=0):.4f}")
print(f"F1 Score:  {f1_score(y_true, y_pred, zero_division=0):.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred, zero_division=0))


CloudSentinel AI — V3 Inference Validation
---------------------------------------------
Test records: 2880
Actual anomalies: 17
Predicted anomalies: 20
Precision: 0.5500
Recall:    0.6471
F1 Score:  0.5946

Confusion Matrix:
[[2854    9]
 [   6   11]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2863
           1       0.55      0.65      0.59        17

    accuracy                           0.99      2880
   macro avg       0.77      0.82      0.80      2880
weighted avg       1.00      0.99      1.00      2880



In [10]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

downloaded_file = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "cloudsentinel_s3_download_test.csv"
)

s3_sample_df = pd.read_csv(downloaded_file)

print("Downloaded sample loaded successfully!")
print("Shape:", s3_sample_df.shape)
print("Columns:", s3_sample_df.columns.tolist())

display(s3_sample_df.head())

Downloaded sample loaded successfully!
Shape: (100, 11)
Columns: ['timestamp', 'resource_id', 'resource_type', 'cpu_utilization', 'memory_utilization', 'network_in', 'request_count', 'storage_usage', 'estimated_cost', 'is_anomaly', 'anomaly_type']


,timestamp,resource_id,resource_type,cpu_utilization,memory_utilization,network_in,request_count,storage_usage,estimated_cost,is_anomaly,anomaly_type
0,2026-08-01 00:00:00,EC2-001,Web Server,19.087236,24.906175,74.708176,899.351067,54.324431,0.059451,0,normal
1,2026-08-01 00:15:00,EC2-001,Web Server,21.372462,28.249815,53.193879,739.607109,56.138281,0.060514,0,normal
2,2026-08-01 00:30:00,EC2-001,Web Server,20.178249,26.371394,57.415241,773.460620,54.555520,0.059651,0,normal
3,2026-08-01 00:45:00,EC2-001,Web Server,16.712111,24.016472,77.400133,922.063367,53.726745,0.058352,0,normal
4,2026-08-01 01:00:00,EC2-001,Web Server,19.658934,25.695524,63.384328,790.523342,54.572166,0.059368,0,normal


In [12]:

from scripts.inference import predict_anomalies

s3_predictions = predict_anomalies(s3_sample_df)

print("CloudSentinel AI — S3 Sample Inference")
print("Records processed:", len(s3_predictions))
print(
    "Predicted anomalies:",
    int(s3_predictions["is_anomaly_prediction"].sum())
)

display(
    s3_predictions[
        [
            "resource_id",
            "cpu_utilization",
            "memory_utilization",
            "estimated_cost",
            "prediction_label",
            "anomaly_score"
        ]
    ].head(10)
)


CloudSentinel AI — S3 Sample Inference
Records processed: 100
Predicted anomalies: 0


,resource_id,cpu_utilization,memory_utilization,estimated_cost,prediction_label,anomaly_score
0,EC2-001,19.087236,24.906175,0.059451,normal,-0.131224
1,EC2-001,21.372462,28.249815,0.060514,normal,-0.126303
2,EC2-001,20.178249,26.371394,0.059651,normal,-0.136469
3,EC2-001,16.712111,24.016472,0.058352,normal,-0.101755
4,EC2-001,19.658934,25.695524,0.059368,normal,-0.136002
5,EC2-001,25.258841,26.619349,0.061663,normal,-0.124369
6,EC2-001,14.293693,25.974754,0.057712,normal,-0.088361
7,EC2-001,25.646439,27.992696,0.062433,normal,-0.129934
8,EC2-001,23.017811,28.649123,0.062359,normal,-0.150878
9,EC2-001,16.415741,27.396574,0.058724,normal,-0.125633


In [15]:

print("Actual anomalies in S3 sample:",
      int(s3_sample_df["is_anomaly"].sum()))

print("\nActual anomaly types:")
display(
    s3_sample_df.loc[
        s3_sample_df["is_anomaly"] == 1,
        ["timestamp", "resource_id", "anomaly_type"]
    ]
)

print("\nPrediction distribution:")
print(s3_predictions["prediction_label"].value_counts())


Actual anomalies in S3 sample: 0

Actual anomaly types:


,timestamp,resource_id,anomaly_type



Prediction distribution:
prediction_label
normal    100
Name: count, dtype: int64


In [14]:

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
RAW_DATA_FILE = (
    PROJECT_ROOT / "data" / "raw" / "cloud_resource_metrics.csv"
)
SAMPLE_OUTPUT = (
    PROJECT_ROOT / "data" / "processed"
    / "cloudsentinel_anomaly_test_sample.csv"
)

full_df = pd.read_csv(RAW_DATA_FILE)

# Select up to 50 injected anomalies and 50 normal records.
anomaly_rows = full_df[full_df["is_anomaly"] == 1].sample(
    n=min(50, int(full_df["is_anomaly"].sum())),
    random_state=42
)
normal_rows = full_df[full_df["is_anomaly"] == 0].sample(
    n=min(50, int((full_df["is_anomaly"] == 0).sum())),
    random_state=42
)

test_sample = (
    pd.concat([anomaly_rows, normal_rows])
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

test_sample.to_csv(SAMPLE_OUTPUT, index=False)

print("Sample saved:", SAMPLE_OUTPUT)
print("Sample shape:", test_sample.shape)
print("\nGround-truth distribution:")
print(test_sample["is_anomaly"].value_counts())


Sample saved: c:\Users\Abhishek\Desktop\CloudSentinel-AI\data\processed\cloudsentinel_anomaly_test_sample.csv
Sample shape: (100, 11)

Ground-truth distribution:
is_anomaly
0    50
1    50
Name: count, dtype: int64


In [16]:

from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)
import pandas as pd

downloaded_sample = pd.read_csv(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "cloudsentinel_anomaly_test_download.csv"
)

sample_predictions = predict_anomalies(downloaded_sample)

y_true = downloaded_sample["is_anomaly"].to_numpy()
y_pred = sample_predictions["is_anomaly_prediction"].to_numpy()

print("Records:", len(downloaded_sample))
print("Actual anomalies:", int(y_true.sum()))
print("Predicted anomalies:", int(y_pred.sum()))
print("Precision:", round(precision_score(y_true, y_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_true, y_pred, zero_division=0), 4))
print("F1 score:", round(f1_score(y_true, y_pred, zero_division=0), 4))
print("\nConfusion matrix:")
print(confusion_matrix(y_true, y_pred))


Records: 100
Actual anomalies: 50
Predicted anomalies: 19
Precision: 1.0
Recall: 0.38
F1 score: 0.5507

Confusion matrix:
[[50  0]
 [31 19]]


In [17]:

import sys
import numpy
import pandas
import sklearn
import joblib

print("Python:", sys.version)
print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Joblib:", joblib.__version__)


Python: 3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]
NumPy: 2.2.6
Pandas: 2.3.3
Scikit-learn: 1.7.2
Joblib: 1.6.0
